## 03_generate_embeddings.ipynb
### Purpose: Pre-compute sentence embeddings for all recipes (one-time cost)

In [1]:
# Cell 1: Imports
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path
import time

print("Libraries loaded.")

c:\Users\admin\Documents\Projects\fridge-to-recipe\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [2]:
# Cell 2: Load cleaned data
processed_path = Path('../processed')
df = pl.read_parquet(processed_path / "recipes_cleaned.parquet")

print(f"Loaded {df.shape[0]:,} recipes")
print("Columns:", df.columns)

Loaded 231,636 recipes
Columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients', 'ingredients_list', 'cleaned_ingredients', 'text_for_embedding']


In [3]:
# Cell 3: Load the embedding model + move to GPU if available
import torch

model_name = "all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

# Force GPU usage (AMD 6800M should work via DirectML or if ROCm is set up)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

if device == "cuda":
    model = model.to(device)
    print("Model moved to GPU - this should be much faster!")

print(f"Model '{model_name}' loaded on {device}.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5761.87it/s]


Using device: CUDA
Model moved to GPU - this should be much faster!
Model 'all-MiniLM-L6-v2' loaded on cuda.


In [4]:
null_rows = df.filter(pl.col("text_for_embedding").is_null())
print(null_rows.shape)
null_rows.select(["id", "name", "ingredients", "description"])

(0, 15)


id,name,ingredients,description
i64,str,str,str


In [5]:
# Cell 4: Generate embeddings (GPU-accelerated)
print("Starting embedding generation...")

start_time = time.time()

embeddings = model.encode(
    df["text_for_embedding"].to_list(),
    batch_size=256,           # Larger batch size is fine on GPU
    show_progress_bar=True,
    convert_to_numpy=True,
    device=device,            # ← This is the key line
    normalize_embeddings=True # Good practice for cosine similarity later
)

end_time = time.time()
print(f"Embeddings generated in {end_time - start_time:.1f} seconds")
print("Embeddings shape:", embeddings.shape)

Starting embedding generation...


Batches: 100%|██████████| 905/905 [00:53<00:00, 17.06it/s]


Embeddings generated in 53.9 seconds
Embeddings shape: (231636, 384)


In [6]:
# Cell 5: Save embeddings + keep a small index file
np.save(processed_path / "recipe_embeddings.npy", embeddings)

# Optional: Save recipe IDs for easy lookup later
recipe_ids = df.select("id").to_numpy().flatten()
np.save(processed_path / "recipe_ids.npy", recipe_ids)

print("Saved:")
print("   - recipe_embeddings.npy")
print("   - recipe_ids.npy")
print("\nYou can now move to the recommender!")

Saved:
   - recipe_embeddings.npy
   - recipe_ids.npy

You can now move to the recommender!
